In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import io
from sklearn.preprocessing import StandardScaler

In [ ]:
from rdkit.Chem import PandasTools
import numpy as np
import pandas as pd
from rdkit import DataStructs
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
import random
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import LeaveOneOut
from sklearn import preprocessing
# from genetic_selection import GeneticSelectionCV
from mordred import Calculator, descriptors

## Cached model loaders
`st.cache_resource` wrappers around `joblib` artefacts.


In [ ]:
@st.cache_resource
def load_models(ficha):
    try:
        if ficha == "a":
            model_a = joblib.load('../streamlit/model_a.pkl')
            return model_a
        if ficha == "b":
            model_b = joblib.load('../streamlit/model_b.pkl')
            return model_b
    except Exception as e:
        st.error(f"Model load error: {e}")
        return None, None

In [ ]:
@st.cache_resource
def load_scalers(ficha):
    try:
        if ficha == "a":
            feature_a = joblib.load('../streamlit/features_a.pkl')
            return feature_a
        if ficha == "b":
            feature_b = joblib.load('../streamlit/features_b.pkl')
            return feature_b
    except:
        return None, None

## Descriptor + reaction parsing
Same Mordred pipeline as production `code.py`.


In [ ]:
def calculate_descriptors_for_molecule(smiles, prefix=""):
    """
    Compute Mordred descriptors for one SMILES string.
    Returns a descriptor dict, or None on failure.
    """
    if pd.isna(smiles):
        return None
        
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
            
        mol_3d = Chem.AddHs(mol)
        Chem.EmbedMolecule(mol_3d, randomSeed=0xf006d)
        
        try:
            Chem.MMFFOptimizeMolecule(mol_3d)
        except:
            pass
            
        calc = Calculator(descriptors)
        desc_dict = calc(mol_3d)
        
        numeric_descriptors = {}
        for key, value in desc_dict.items():
            try:
                float_value = float(value)
                numeric_descriptors[f"{prefix}{key}"] = float_value
            except (ValueError, TypeError):
                continue
                
        return numeric_descriptors
        
    except Exception as e:
        print(f"Error for SMILES {smiles}: {e}")
        return None

In [ ]:
def process_reactions(data):
    """
    Process all reactions and return descriptor rows.
    """
    all_descriptors = []
    
    for idx, reaction_smiles in enumerate(data['SMILES']):
        if pd.isna(reaction_smiles):
            all_descriptors.append({})
            continue
        
        print(f"Processing reaction {idx+1}/{len(data)}: {reaction_smiles}")
        
        reaction_descriptors = {}
        
        if '>>' in str(reaction_smiles):
            reagents_part, products_part = reaction_smiles.split('>>')
            
            reagents = [r.strip() for r in reagents_part.split('.') if r.strip()]
            for i, reagent in enumerate(reagents):
                prefix = f"reagent{i+1}_"
                desc_dict = calculate_descriptors_for_molecule(reagent, prefix)
                if desc_dict:
                    reaction_descriptors.update(desc_dict)
            
            products = [p.strip() for p in products_part.split('.') if p.strip()]
            for i, product in enumerate(products):
                prefix = f"product{i+1}_"
                desc_dict = calculate_descriptors_for_molecule(product, prefix)
                if desc_dict:
                    reaction_descriptors.update(desc_dict)
        
        all_descriptors.append(reaction_descriptors)

    print ("Processing finished successfully")
    
    return all_descriptors

In [ ]:
def preprocess_data(raw_df, ficha, n):
    try:
        all_desc_data = process_reactions(raw_df)
        features = load_scalers(ficha)
        return all_desc_data[features[:n]]
    except Exception as e:
        st.error(f"Data processing error: {e}")
        return None

## Streamlit `main`
Prototype UI: upload, process, predict, download CSV.


In [ ]:
def main():
    st.title("📊 ML Models with SMILES Upload")
    st.write("Upload a data file for prediction.")
    
    st.sidebar.header("Model settings")
    
    target_choice = st.sidebar.selectbox(
        "Select prediction target:",
        ["a", "b"],
        help="Labels follow the ChemDraw naming scheme."
    )

    model = load_models(target_choice)    
    if model is None:
        st.error("Models failed to load. Check model files.")
        return
    
    st.header("1. Upload SMILES file")
    uploaded_file = st.file_uploader(
        "Choose data file", 
        type=['csv', 'xlsx', 'xls'],
        help="Supported formats: CSV and Excel. The file must include a column named 'SMILES'. " \
        "Each SMILES cell must contain the full reaction (reactants>>products)."
    )
    
    raw_data = None
    processed_data = None
    
    if uploaded_file is not None:
        try:
            if uploaded_file.name.endswith('.csv'):
                raw_data = pd.read_csv(uploaded_file)
            else:
                raw_data = pd.read_excel(uploaded_file)
            
            st.success(f"File {uploaded_file.name} loaded successfully.")
            
            #     st.dataframe(raw_data.head())
                
            #     st.write(raw_data.describe())
            
            st.header("2. Обработка данных")
            if st.button("Process data", type="primary"):
                with st.spinner("Processing data..."):
                    processed_data = preprocess_data(raw_data, target_choice, 30)
                    
                    if processed_data is not None:
                        with st.expander("🔧 Processed data preview"):
                            st.write(f"Shape after processing: {processed_data.shape[0]} rows, {processed_data.shape[1]} columns")
                            st.dataframe(processed_data.head())
            
        except Exception as e:
            st.error(f"File load error: {e}")
    
    st.header("3. Предсказание")
    
    if processed_data is not None:
        
        if st.button("🚀 Run prediction", type="primary"):
            try:
                with st.spinner("Running prediction..."):

                    predictions = model.predict(processed_data)

                    st.success("Prediction finished.")
                    
                    results_df = processed_data.copy()
                    results_df['Predicted_Class'] = predictions
                    
                    st.subheader("📈 Prediction results")
                    st.dataframe(results_df[['SMILES', 'Predicted_Class']])
                    
                    st.subheader("📊 Статистика предсказаний")
                    pred_counts = results_df['Predicted_Class'].value_counts()
                    st.write(pred_counts)
                    
                    st.bar_chart(pred_counts)
                    
                    csv = results_df.to_csv(index=False)
                    st.download_button(
                        label="📥 Download results (CSV)",
                        data=csv,
                        file_name=f"smiles_predictions_{target_choice}.csv",
                        mime="text/csv"
                    )
                    
                    
            except Exception as e:
                st.error(f"Prediction error: {e}")
                st.info("Ensure the input table matches the model's expected schema.")
    
    else:
        st.info("👆 Upload a file and run processing to obtain predictions.")

In [ ]:
if __name__ == "__main__":
    main()